# OneShot FaceID: face verification with a Siamese network

End-to-end walkthrough: collect data → build the Siamese network → train → evaluate → verify in real time.

Based on *Siamese Neural Networks for One-shot Image Recognition* (Koch et al., 2015) and Nicholas Renotte's
"Build a Deep Facial Recognition App" series, refactored into reusable modules in `src/`.

> Run this notebook from the `notebooks/` folder. Personal images and trained weights are git-ignored.

## 1. Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import cv2
import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt

from src import config
from src.data import make_dirs, move_lfw_to_negatives, preprocess, build_datasets
from src.model import make_embedding, make_siamese_model, load_siamese_model
from src.train import train, evaluate

# Avoid out-of-memory errors: let TensorFlow grow GPU memory as needed
for gpu in tf.config.experimental.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

make_dirs()

## 2. Collect data

**Negatives** – faces of other people from *Labelled Faces in the Wild* (LFW). Download and unzip it, then flatten the
one-folder-per-person structure into `data/negative`:

In [ ]:
# Path to the unzipped LFW folder that contains one sub-folder per person
LFW_ROOT = '../lfw-deepfunneled'
if os.path.isdir(LFW_ROOT):
    print('Moved', move_lfw_to_negatives(LFW_ROOT), 'images to', config.NEG_PATH)

**Anchors and positives** – your own face from the webcam (250×250 crop). Run `python -m src.collect` from the repo root
and press `a` for anchors, `p` for positives, `q` to quit. Aim for ~300 of each, with different angles and lighting.

In [ ]:
for name, folder in [('anchor', config.ANC_PATH), ('positive', config.POS_PATH), ('negative', config.NEG_PATH)]:
    print(f'{name:9s}', len(list(folder.glob('*.jpg'))), 'images')

### 2b. Augment anchors and positives
Each original photo gets 9 variants (brightness, contrast, mirror, JPEG quality, saturation). Every variant starts from the original image with fresh random seeds, so ~300 originals become ~3,000 images per class.

In [ ]:
from src.augment import augment_folder
for folder in (config.ANC_PATH, config.POS_PATH):
    print(folder.name, '->', augment_folder(folder, n=9), 'augmented images')

## 3. Preprocess and build the pipeline

In [ ]:
sample = next(config.ANC_PATH.glob('*.jpg'), None)
if sample:
    img = preprocess(str(sample))
    print('shape', img.shape, '| pixel range', float(tf.reduce_min(img)), '-', float(tf.reduce_max(img)))
    plt.imshow(img); plt.axis('off'); plt.show()

Pairs: *(anchor, positive) → 1* and *(anchor, negative) → 0*. 70% train / 30% test, batches of 16.

In [ ]:
train_data, test_data = build_datasets()
print('train batches:', len(train_data), '| test batches:', len(test_data))

## 4. Model

In [ ]:
embedding = make_embedding()
embedding.summary()

In [ ]:
siamese_model = make_siamese_model(embedding)
siamese_model.summary()

## 5. Train
Custom `GradientTape` loop, binary cross-entropy, Adam (1e-4), checkpoint every 10 epochs.

In [ ]:
history = train(siamese_model, train_data, epochs=config.EPOCHS)

In [ ]:
epochs = [h['epoch'] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, [h['loss'] for h in history]); ax[0].set_title('Training loss')
ax[1].plot(epochs, [h['precision'] for h in history], label='precision')
ax[1].plot(epochs, [h['recall'] for h in history], label='recall'); ax[1].legend(); ax[1].set_title('Training precision / recall')
plt.show()

## 6. Evaluate on the held-out set
Metrics are computed over the whole test set, not a single batch.

In [ ]:
metrics = evaluate(siamese_model, test_data, threshold=0.5)
metrics

In [ ]:
test_input, test_val, y_true = test_data.as_numpy_iterator().next()
y_hat = siamese_model.predict([test_input, test_val], verbose=0).ravel()

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    axes[0, i].imshow(test_input[i]); axes[0, i].axis('off')
    axes[1, i].imshow(test_val[i]);   axes[1, i].axis('off')
    axes[0, i].set_title(f'true={int(y_true[i])}  p={y_hat[i]:.2f}', fontsize=9)
plt.suptitle('Top: input image · Bottom: comparison image'); plt.show()

## 7. Save and reload

In [ ]:
config.MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
siamese_model.save(config.MODEL_PATH)
reloaded = load_siamese_model(str(config.MODEL_PATH))
print('Reloaded predictions match:', np.allclose(reloaded.predict([test_input, test_val], verbose=0).ravel(), y_hat, atol=1e-5))

## 8. Real-time verification

1. Copy 30–50 of your positive images into `application_data/verification_images`.
2. Run `python -m src.verify` (OpenCV, press `v`) or `python -m app.faceid` (Kivy desktop app).

A frame is **verified** when more than 50% of the reference photos score above 0.5. Both thresholds are tunable:
a stricter detection threshold lowers false accepts at the cost of more false rejects.